# Attention-backend benchmark (T4) — all models

Times every model with **eager vs sdpa** (the backends available on a T4). **flash_attention_2 needs Ampere+; it does NOT run on a T4/Turing GPU**, so it's opt-in and will show FAIL here — that's expected. InternVL's remote code rejects sdpa (records FAIL); everything else should run both.

Runtime → T4 GPU → Run all.


## 1. GPU + setup

In [ ]:
!nvidia-smi -L

In [ ]:
%cd /content
![ -d OCR ] || git clone https://github.com/SangbumChoi/OCR.git


In [ ]:
%cd /content/OCR
!git checkout claude/new-session-w79q0i && git pull --ff-only

In [ ]:
!pip -q install -e '.[models]'
!pip -q install 'transformers==4.49.0' peft protobuf

## 2. Build probe + bench ALL chat VLMs (eager vs sdpa)
InternVL/Florence/Ovis/H2OVL/GOT/SmolVLM/LLaVA/SmolDocling — every model, both backends.

In [ ]:
%cd /content/OCR
!python scripts/make_capability_probe.py
!python scripts/bench_attention.py --device cuda --attns eager sdpa \
   --models internvl2-1b internvl2_5-1b internvl3-1b smolvlm-256m smolvlm-500m \
            smoldocling-256m llava-ov-0.5b got-ocr2 florence2-base florence2-large \
            h2ovl-0.8b ovis2-1b \
   --out docs/results/attention_benchmark_chat.md

In [ ]:
print(open('/content/OCR/results/attention_benchmark_chat.md').read())

## 3. PaddleOCR-VL (needs transformers 4.57 — separate pass)

In [ ]:
%cd /content/OCR
!pip -q install 'transformers==4.57.1' protobuf
!python scripts/bench_attention.py --device cuda --attns eager sdpa \
   --models paddleocr-vl paddleocr-vl-1.5 paddleocr-vl-1.6 \
   --out docs/results/attention_benchmark_paddle.md

In [ ]:
print(open('/content/OCR/results/attention_benchmark_paddle.md').read())

## 4. (Optional) try flash-attention 2 — only meaningful on Ampere+ (A100/L4), not T4
On a T4 this will FAIL (Turing unsupported). Run on an A100/L4 runtime to compare.

In [ ]:
%cd /content/OCR
# flash-attn build is slow and Turing-unsupported; only attempt on Ampere+
!pip -q install flash-attn --no-build-isolation || echo 'flash-attn unavailable'
!python scripts/bench_attention.py --device cuda --attns eager sdpa flash_attention_2 \
   --models smolvlm-500m internvl2_5-1b --out docs/results/attention_benchmark_flash.md
print(open('docs/results/attention_benchmark_flash.md').read())

## 5. Commit results back (so the assistant can read them)

In [ ]:
%cd /content/OCR
!git add -f docs/results/attention_benchmark_*.md && \
 git -c user.email=colab@local -c user.name=colab commit -m 'attn benchmark results' && \
 git push origin claude/new-session-w79q0i